# Acrobot DQN — Google Colab
Deep Q-Network agent trained on the `Acrobot-v1` environment using Gymnasium and TensorFlow.

In [ ]:
# Install dependencies
!pip install gymnasium[classic-control] tensorflow tqdm
!apt-get install -y ffmpeg

In [ ]:
import gymnasium as gym
from gymnasium.wrappers import RecordVideo
import tensorflow as tf
from collections import deque
import numpy as np
import random
from tqdm import tqdm

In [ ]:
class GymEnvironment:
    def __init__(self, env_id, max_timesteps=120, render_mode=None):
        self.max_timesteps = max_timesteps
        self.env = gym.make(env_id, render_mode=render_mode)

    def trainDQN(self, agent):
        rew_hist, loss = self.runDQN(agent, training=True)
        agent.model.save_weights('tmp.h5', overwrite=True)
        return rew_hist, loss

    def runDQN(self, agent, training=False):
        rew_hist = []
        loss = []
        max_no_episodes = 300  # Reduced from 1000 for Colab speed

        for episode in range(max_no_episodes):
            state = self.env.reset()[0].reshape(1, self.env.observation_space.shape[0])
            total_reward = 0
            done = False
            t = 0

            while not done and t < self.max_timesteps:
                action = agent.select_action(state, explore=training)
                next_state, reward, done, _, __ = self.env.step(action)
                next_state = next_state.reshape(1, self.env.observation_space.shape[0])

                if training:
                    agent.record(state, next_state, done, reward, action)
                    agent.update_weights()

                state = next_state
                total_reward += reward
                t += 1

            rew_hist.append(total_reward)

            if (episode + 1) % 50 == 0:
                print("episode: {}/{} | score: {} | e: {:.3f}".format(
                    episode + 1, max_no_episodes, total_reward, agent.epsilon))

            if agent.epsilon > agent.epsilon_min:
                agent.epsilon *= agent.epsilon_decay

        return rew_hist, loss

    def testDQN(self, agent):
        total_episodes = 100
        total_rewards = []

        for episode in range(total_episodes):
            state = self.env.reset()[0].reshape(1, self.env.observation_space.shape[0])
            total_reward = 0
            done = False

            while not done:
                action = agent.select_action(state, explore=False)
                next_state, reward, done, _, __ = self.env.step(action)
                next_state = next_state.reshape(1, self.env.observation_space.shape[0])
                state = next_state
                total_reward += reward

            total_rewards.append(total_reward)

        avg_reward = np.mean(total_rewards)
        print("Average test reward over {} episodes: {:.2f}".format(total_episodes, avg_reward))
        return avg_reward

In [ ]:
class DQN_Agent:
    def __init__(self, no_of_states, no_of_actions, load_old_model=False):
        self.state_vector_size = no_of_states
        self.action_space_size = no_of_actions

        self.gamma = 0.9
        self.epsilon = 1.0          # Fixed: was 10, must be in [0, 1]
        self.epsilon_min = 0.01
        self.epsilon_decay = 0.995

        self.model = self.nn_model(load_old_model)
        self.target_model = self.nn_model(load_old_model=False)
        self.target_model.set_weights(self.model.get_weights())

        self.replay_buffer = deque(maxlen=5000)

    def nn_model(self, load_old_model=False):
        if load_old_model:
            model = tf.keras.models.load_model('tmp.h5')
        else:
            model = tf.keras.Sequential([
                tf.keras.layers.Dense(128, input_shape=(self.state_vector_size,), activation='relu'),
                tf.keras.layers.Dense(128, activation='relu'),
                tf.keras.layers.Dense(self.action_space_size, activation='linear')
            ])
            model.compile(loss='mse', optimizer=tf.keras.optimizers.Adam(learning_rate=0.001))
        return model

    def select_action(self, state, explore=True):
        if explore and np.random.rand() <= self.epsilon:
            return np.random.choice(self.action_space_size)
        state_tensor = tf.convert_to_tensor(state, dtype=tf.float32)
        q_values = self.model(state_tensor)
        return int(tf.argmax(q_values[0]))

    def record(self, state, next_state, done, reward, action):
        self.replay_buffer.append((state, np.array(next_state), done, reward, action))

    def update_weights(self):
        if len(self.replay_buffer) < 32:
            return

        batch = random.sample(self.replay_buffer, 32)
        state_batch, next_state_batch, done_batch, reward_batch, action_batch = zip(*batch)

        state_batch = np.vstack(state_batch)
        next_state_batch = np.vstack(next_state_batch)
        target = self.model.predict(state_batch, verbose=0)
        target_next = self.target_model.predict(next_state_batch, verbose=0)

        for i in range(len(batch)):
            if done_batch[i]:
                target[i][action_batch[i]] = reward_batch[i]
            else:
                target[i][action_batch[i]] = reward_batch[i] + self.gamma * np.amax(target_next[i])

        self.model.fit(state_batch, target, epochs=1, verbose=0)

    def update_target_weights(self):
        self.target_model.set_weights(self.model.get_weights())

In [ ]:
# Train the agent
environment = GymEnvironment('Acrobot-v1')
state_vector_size = environment.env.observation_space.shape[0]
action_space_size = environment.env.action_space.n

total_iterations = 3  # Increase for better performance

for i in tqdm(range(total_iterations), desc="Training"):
    agent = DQN_Agent(state_vector_size, action_space_size)
    rew_hist, loss = environment.trainDQN(agent)
    rew_hist, loss = environment.runDQN(agent)
    environment.testDQN(agent)

environment.env.close()
print("Training complete. Weights saved to tmp.h5")

In [ ]:
# Visualize the trained agent and record a video
import os
os.makedirs('./videos', exist_ok=True)

vis_env = gym.make('Acrobot-v1', render_mode='rgb_array')
vis_env = RecordVideo(vis_env, video_folder='./videos', episode_trigger=lambda e: True)

agent = DQN_Agent(state_vector_size, action_space_size, load_old_model=True)

state = vis_env.reset()[0].reshape(1, state_vector_size)
done = False
total_reward = 0

while not done:
    action = agent.select_action(state, explore=False)
    next_state, reward, done, _, __ = vis_env.step(action)
    state = next_state.reshape(1, state_vector_size)
    total_reward += reward

vis_env.close()
print(f"Episode reward: {total_reward}")

In [ ]:
# Play the recorded video inline
from IPython.display import Video
import glob

video_files = sorted(glob.glob('./videos/*.mp4'))
Video(video_files[-1], embed=True, width=500)